# C2.1 Production RAG Architecture

This lesson is production-first: decide when to use RAG, build a corpus, evaluate chunking and retrieval, and assemble a grounded RAG pipeline.

Learning goals:
- Choose RAG vs fine-tuning with clear tradeoffs.
- Compare chunking strategies with retrieval metrics.
- Implement query rewriting and measure its impact.
- Build a complete RAG pipeline without external downloads.

## RAG vs fine-tuning decision

**Direct answer for this program:** use RAG for knowledge-grounded applications. Fine-tuning changes behavior, not knowledge.

Why RAG dominates production for knowledge-grounded apps:
- Retrieves fresh, private, and auditable data at request time.
- Keeps model weights stable while knowledge changes frequently.
- Reduces hallucinations by grounding responses in retrieved evidence.

Why fine-tuning is often impractical for product teams without training infra:
- Requires labeled data, GPUs, and training pipelines.
- Updates are slow and expensive when knowledge changes.
- Hard to trace or cite sources for answers.

Tradeoffs (cost, maintenance, recency):

| Dimension | RAG | Fine-tuning |
| --- | --- | --- |
| Cost | Low upfront, higher per-query | High upfront, lower per-query |
| Maintenance | Update index quickly | Retrain to update knowledge |
| Recency | Immediate | Delayed until retrain |

## Chunking strategies

No single method is universally best. Choose based on data type, retrieval goals, and system architecture.

Typical fits:
- Sentence-aware (recursive/sentence-level) for general-purpose RAG when you want coherent context without heavy compute.
- Semantic chunking for topic-heavy, long documents where precision matters and extra compute is acceptable.
- Fixed-size (sliding window) for high-throughput pipelines where predictability matters most.

Detailed breakdown:

1. Sentence-aware (recursive/sentence-level)
- What it does: splits text by logical units like sentences or paragraphs.
- Best fit: standard RAG use cases that need coherent context.
- Pros: preserves complete thoughts and avoids mid-sentence breaks.
- Cons: inconsistent chunk sizes can affect models expecting uniform inputs.

2. Semantic chunking
- What it does: uses embeddings or NLP similarity to split when topics shift.
- Best fit: long research papers, dense manuals, or multi-topic documents.
- Pros: high retrieval precision because chunks stay concept-cohesive.
- Cons: more complex and compute-heavy because it requires embedding to chunk.

3. Fixed size (sliding window)
- What it does: splits at a fixed token or character count with overlap.
- Best fit: logs, structured reports, or large datasets where throughput matters.
- Pros: simple, fast, and predictable latency.
- Cons: breaks sentences and can dilute semantic meaning.

Tradeoff snapshot (no universal winner):

| Method | Coherence | Compute | Latency predictability | Typical fit |
| --- | --- | --- | --- | --- |
| Sentence-aware | High | Medium | Medium | General RAG |
| Semantic | Highest | High | Low | Topic-heavy docs |
| Fixed-size | Low to Medium | Low | High | High-throughput |

Chunk size tradeoff:
- Too small: loses context and harms retrieval quality.
- Too large: mixes topics and adds irrelevant context.
- Balanced sizes preserve meaning and keep retrieval precise.

In [39]:
from pathlib import Path
import csv
import json
import math
import re

import numpy as np

In [40]:
DATA_DIR = Path('data') / 'C2.1 corpus'
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Clean previous files to keep counts stable
for path in DATA_DIR.iterdir():
    if path.is_file():
        path.unlink()

base_facts = [
    {
        'topic': 'ai',
        'title': 'Backpropagation and gradients',
        'body': 'Backpropagation computes gradients for neural networks during training.',
        'query': 'What computes gradients for neural networks?',
        'answer': 'backpropagation'
    },
    {
        'topic': 'ai',
        'title': 'Dropout regularization',
        'body': 'Dropout reduces overfitting by randomly dropping units during training.',
        'query': 'Which technique reduces overfitting by randomly dropping units?',
        'answer': 'dropout'
    },
    {
        'topic': 'climate',
        'title': 'Greenhouse gases',
        'body': 'Carbon dioxide is a key greenhouse gas that traps heat.',
        'query': 'Which gas is a key greenhouse gas?',
        'answer': 'carbon dioxide'
    },
    {
        'topic': 'climate',
        'title': 'Renewable energy',
        'body': 'Solar and wind are renewable energy sources that reduce fossil fuel use.',
        'query': 'Name two renewable energy sources mentioned.',
        'answer': 'solar and wind'
    },
    {
        'topic': 'travel',
        'title': 'Taj Mahal origin',
        'body': 'The Taj Mahal was built by Emperor Shah Jahan in Agra.',
        'query': 'Who built the Taj Mahal?',
        'answer': 'Shah Jahan'
    },
    {
        'topic': 'travel',
        'title': 'Eiffel Tower location',
        'body': 'The Eiffel Tower is in Paris, France and is a major landmark.',
        'query': 'Where is the Eiffel Tower located?',
        'answer': 'Paris'
    },
    {
        'topic': 'health',
        'title': 'Vaccines and immunity',
        'body': 'Vaccines train the immune system to recognize specific pathogens.',
        'query': 'What do vaccines train?',
        'answer': 'immune system'
    },
    {
        'topic': 'health',
        'title': 'Antibiotics limits',
        'body': 'Antibiotics treat bacterial infections and do not treat viruses.',
        'query': 'What do antibiotics not treat?',
        'answer': 'viruses'
    },
    {
        'topic': 'finance',
        'title': 'Inflation basics',
        'body': 'Inflation reduces purchasing power over time.',
        'query': 'What reduces purchasing power over time?',
        'answer': 'inflation'
    },
    {
        'topic': 'finance',
        'title': 'Diversification',
        'body': 'Diversification reduces portfolio risk by spreading exposure.',
        'query': 'What reduces portfolio risk?',
        'answer': 'diversification'
    },
    {
        'topic': 'sports',
        'title': 'World Cup schedule',
        'body': 'The FIFA World Cup is held every four years.',
        'query': 'How often is the FIFA World Cup held?',
        'answer': 'every four years'
    },
    {
        'topic': 'sports',
        'title': 'Major basketball league',
        'body': 'The NBA is a major basketball league in the United States.',
        'query': 'Which league is a major basketball league in the United States?',
        'answer': 'NBA'
    }
]

topic_filler = {
    'ai': [
        'Neural networks learn patterns from data.',
        'Model capacity grows with more parameters.',
        'Validation data helps detect overfitting.',
        'Feature scaling can improve optimization.'
    ],
    'climate': [
        'Sea level rise affects coastal regions.',
        'Energy efficiency reduces total demand.',
        'Reforestation can absorb atmospheric carbon.',
        'Extreme weather events can increase costs.'
    ],
    'travel': [
        'Local cuisine is part of cultural tourism.',
        'Historic sites often require preservation.',
        'Public transit helps visitors navigate cities.',
        'Travel guides provide seasonal tips.'
    ],
    'health': [
        'Sleep supports recovery and cognitive function.',
        'Exercise improves cardiovascular health.',
        'Hydration supports normal metabolism.',
        'Preventive care reduces long term risk.'
    ],
    'finance': [
        'Bonds provide fixed income payments.',
        'Compounding grows savings over time.',
        'Liquidity measures how quickly assets convert to cash.',
        'Budgeting improves cash flow planning.'
    ],
    'sports': [
        'Teamwork improves performance under pressure.',
        'Training intensity should increase gradually.',
        'Strategy changes based on opponent strengths.',
        'Recovery days prevent injury.'
    ]
}

documents = []
corpus_queries = []
doc_id = 1

for fact in base_facts:
    documents.append({
        'id': doc_id,
        'title': fact['title'],
        'body': fact['body'],
        'topic': fact['topic']
    })
    corpus_queries.append({
        'query': fact['query'],
        'doc_id': doc_id,
        'answer': fact['answer']
    })
    doc_id += 1

filler_per_topic = 8

for topic, sentences in topic_filler.items():
    for i in range(filler_per_topic):
        title = f"{topic.title()} Notes {i + 1:02d}"
        body = ' '.join([
            sentences[i % len(sentences)],
            sentences[(i + 1) % len(sentences)]
        ])
        documents.append({
            'id': doc_id,
            'title': title,
            'body': body,
            'topic': topic
        })
        doc_id += 1

formats = ['txt'] * 15 + ['md'] * 15 + ['jsonl'] * 15 + ['csv'] * 15
for doc, fmt in zip(documents, formats):
    doc['format'] = fmt

for doc in documents:
    if doc['format'] == 'txt':
        path = DATA_DIR / f"doc_{doc['id']:03d}.txt"
        path.write_text(
            'Title: {title}\n{body}\n'.format(
                title=doc['title'],
                body=doc['body']
            ),
            encoding='utf-8'
        )
    elif doc['format'] == 'md':
        path = DATA_DIR / f"doc_{doc['id']:03d}.md"
        path.write_text(
            '# {title}\n\n{body}\n'.format(
                title=doc['title'],
                body=doc['body']
            ),
            encoding='utf-8'
        )

jsonl_docs = [d for d in documents if d['format'] == 'jsonl']
jsonl_path = DATA_DIR / 'docs.jsonl'
with jsonl_path.open('w', encoding='utf-8') as f:
    for doc in jsonl_docs:
        f.write(json.dumps({
            'id': doc['id'],
            'title': doc['title'],
            'body': doc['body']
        }) + '\n')

csv_docs = [d for d in documents if d['format'] == 'csv']
csv_path = DATA_DIR / 'docs.csv'
with csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'title', 'body'])
    writer.writeheader()
    for doc in csv_docs:
        writer.writerow({
            'id': doc['id'],
            'title': doc['title'],
            'body': doc['body']
        })

corpus_documents = documents

format_counts = {}
for doc in documents:
    format_counts[doc['format']] = format_counts.get(doc['format'], 0) + 1

print('Corpus created.')
print(f'Total documents: {len(documents)}')
print(f'Formats: {format_counts}')
print(f'Data folder: {DATA_DIR.resolve()}')
print(f'Query set size: {len(corpus_queries)}')

Corpus created.
Total documents: 60
Formats: {'txt': 15, 'md': 15, 'jsonl': 15, 'csv': 15}
Data folder: D:\AKASH.O\Documents\Study\AI-Engineering\data\C2.1 corpus
Query set size: 12


The output confirms a 60-document corpus with 15 files per format and 12 evaluation queries, so the lab has enough data and multiple formats for ingestion testing.

In [41]:
def load_txt(path):
    text = path.read_text(encoding='utf-8')
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    title = lines[0].replace('Title: ', '') if lines else 'Untitled'
    body = ' '.join(lines[1:]) if len(lines) > 1 else ''
    return title, body

def load_md(path):
    text = path.read_text(encoding='utf-8')
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    title = lines[0].replace('# ', '') if lines else 'Untitled'
    body = ' '.join(lines[1:]) if len(lines) > 1 else ''
    return title, body

def load_jsonl(path):
    docs = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            docs.append({
                'id': int(item['id']),
                'title': item['title'],
                'body': item['body'],
                'source': 'jsonl'
            })
    return docs

def load_csv(path):
    docs = []
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            docs.append({
                'id': int(row['id']),
                'title': row['title'],
                'body': row['body'],
                'source': 'csv'
            })
    return docs

def load_corpus(data_dir):
    docs = []
    for path in data_dir.glob('*.txt'):
        title, body = load_txt(path)
        docs.append({
            'id': int(path.stem.split('_')[1]),
            'title': title,
            'body': body,
            'source': 'txt'
        })
    for path in data_dir.glob('*.md'):
        title, body = load_md(path)
        docs.append({
            'id': int(path.stem.split('_')[1]),
            'title': title,
            'body': body,
            'source': 'md'
        })
    for path in data_dir.glob('*.jsonl'):
        docs.extend(load_jsonl(path))
    for path in data_dir.glob('*.csv'):
        docs.extend(load_csv(path))
    docs.sort(key=lambda d: d['id'])
    return docs

In [42]:
ingested_docs = load_corpus(DATA_DIR)

counts = {}
for doc in ingested_docs:
    counts[doc['source']] = counts.get(doc['source'], 0) + 1

print(f'Ingested documents: {len(ingested_docs)}')
print(f'By format: {counts}')
print(f"Example title: {ingested_docs[0]['title']}")

Ingested documents: 60
By format: {'txt': 15, 'md': 15, 'jsonl': 15, 'csv': 15}
Example title: Backpropagation and gradients


Ingestion loaded all 60 documents across four formats, and the sample title shows the parser is reading each file type correctly.

## Lab 1: Chunking baseline vs improved

Goal: use fixed-size chunking as a baseline and measure how sentence-aware and semantic chunking change retrieval quality on the same corpus. This is not about a single winner, but about tradeoffs and improvements vs baseline.

In [43]:
def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())


class SimpleTfidfVectorizer:
    def __init__(self):
        self.vocab = {}
        self.idf = None

    def fit(self, texts):
        doc_count = len(texts)
        df = {}
        for text in texts:
            terms = set(tokenize(text))
            for term in terms:
                df[term] = df.get(term, 0) + 1
        self.vocab = {term: idx for idx, term in enumerate(sorted(df.keys()))}
        self.idf = np.zeros(len(self.vocab), dtype=np.float32)
        for term, idx in self.vocab.items():
            self.idf[idx] = math.log((1 + doc_count) / (1 + df[term])) + 1.0
        return self

    def transform(self, texts):
        vectors = np.zeros((len(texts), len(self.vocab)), dtype=np.float32)
        if not self.vocab:
            return vectors
        for i, text in enumerate(texts):
            terms = tokenize(text)
            if not terms:
                continue
            counts = {}
            for term in terms:
                counts[term] = counts.get(term, 0) + 1
            for term, count in counts.items():
                idx = self.vocab.get(term)
                if idx is None:
                    continue
                tf = count / len(terms)
                vectors[i, idx] = tf * self.idf[idx]
        return vectors

    def fit_transform(self, texts):
        self.fit(texts)
        return self.transform(texts)


def split_sentences(text):
    text = text.strip()
    if not text:
        return []
    return re.split(r'(?<=[.!?])\s+', text)


def fixed_size_chunking(text, chunk_size=40, overlap=5):
    words = text.split()
    chunks = []
    step = max(chunk_size - overlap, 1)
    for i in range(0, len(words), step):
        chunk = words[i:i + chunk_size]
        if chunk:
            chunks.append(' '.join(chunk))
    return chunks


def sentence_aware_chunking(text, max_words=60, overlap_sentences=1):
    sentences = split_sentences(text)
    chunks = []
    current_chunk = []
    current_len = 0
    for sentence in sentences:
        sentence_len = len(sentence.split())
        if current_len + sentence_len > max_words and current_chunk:
            chunks.append(' '.join(current_chunk))
            overlap = current_chunk[-overlap_sentences:]
            current_chunk = overlap.copy()
            current_len = sum(len(s.split()) for s in current_chunk)
        current_chunk.append(sentence)
        current_len += sentence_len
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks


def cosine_similarity(vec_a, vec_b):
    denom = (np.linalg.norm(vec_a) * np.linalg.norm(vec_b)) + 1e-10
    return float(np.dot(vec_a, vec_b) / denom)


def semantic_chunking(text, similarity_threshold=0.25, min_sentences=2, max_words=80):
    sentences = split_sentences(text)
    if not sentences:
        return []
    vectorizer = SimpleTfidfVectorizer()
    vectors = vectorizer.fit_transform(sentences)
    chunks = []
    current_chunk = [sentences[0]]
    current_len = len(sentences[0].split())
    for i in range(1, len(sentences)):
        similarity = cosine_similarity(vectors[i - 1], vectors[i])
        sentence_len = len(sentences[i].split())
        should_append = (
            similarity >= similarity_threshold
            or len(current_chunk) < min_sentences
        )
        if should_append and current_len + sentence_len <= max_words:
            current_chunk.append(sentences[i])
            current_len += sentence_len
        else:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentences[i]]
            current_len = sentence_len
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks

In [44]:
def build_chunks(docs, chunk_fn):
    chunks = []
    for doc in docs:
        text = f"{doc['title']}. {doc['body']}"
        for chunk in chunk_fn(text):
            chunks.append({
                'doc_id': doc['id'],
                'title': doc['title'],
                'text': chunk
            })
    return chunks


def cosine_similarity_scores(query_vec, doc_vecs):
    query_norm = np.linalg.norm(query_vec) + 1e-10
    doc_norms = np.linalg.norm(doc_vecs, axis=1) + 1e-10
    return (doc_vecs @ query_vec) / (doc_norms * query_norm)


def retrieve_top_k(query, chunks, vectorizer, chunk_vectors, top_k=3):
    query_vec = vectorizer.transform([query])[0]
    scores = cosine_similarity_scores(query_vec, chunk_vectors)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [chunks[i] for i in top_indices]


def evaluate_chunking(method_name, chunk_fn, docs, queries, top_k=3):
    chunks = build_chunks(docs, chunk_fn)
    avg_len = float(np.mean([len(c['text'].split()) for c in chunks]))
    vectorizer = SimpleTfidfVectorizer()
    chunk_vectors = vectorizer.fit_transform([c['text'] for c in chunks])
    top1 = 0
    top3 = 0
    mrr_scores = []
    for item in queries:
        retrieved = retrieve_top_k(item['query'], chunks, vectorizer, chunk_vectors, top_k=top_k)
        answer = item['answer'].lower()
        found_rank = None
        for idx, chunk in enumerate(retrieved):
            if answer in chunk['text'].lower():
                found_rank = idx + 1
                break
        if found_rank == 1:
            top1 += 1
        if found_rank is not None and found_rank <= top_k:
            top3 += 1
        mrr_scores.append(1.0 / found_rank if found_rank else 0.0)
    total = len(queries)
    return {
        'method': method_name,
        'chunks': len(chunks),
        'avg_len': avg_len,
        'top1': top1 / total if total else 0.0,
        'top3': top3 / total if total else 0.0,
        'mrr': float(np.mean(mrr_scores)) if mrr_scores else 0.0
    }

In [45]:
fixed_chunker = lambda text: fixed_size_chunking(text, chunk_size=8, overlap=0)
sentence_chunker = lambda text: sentence_aware_chunking(text, max_words=40, overlap_sentences=1)
semantic_chunker = lambda text: semantic_chunking(text, similarity_threshold=0.35, min_sentences=2, max_words=60)

comparison_docs = [
    {
        'id': 201,
        'title': 'Gradients and climate',
        'body': (
            'Backpropagation computes gradients for neural networks during training. '
            'Regularization techniques can reduce overfitting during training. '
            'Carbon dioxide is a key greenhouse gas that traps heat. '
            'Sea level rise affects coastal regions.'
        )
    },
    {
        'id': 202,
        'title': 'Training optimization notes',
        'body': (
            'Gradient descent updates neural network weights during training. '
            'Neural networks learn patterns from data. '
            'Optimization improves convergence for neural networks.'
        )
    },
    {
        'id': 203,
        'title': 'Renewable energy and finance',
        'body': (
            'Solar and wind are renewable energy sources. '
            'They reduce fossil fuel use in power grids. '
            'Inflation reduces purchasing power over time. '
            'Budgeting improves cash flow planning.'
        )
    },
    {
        'id': 204,
        'title': 'Energy policy brief',
        'body': (
            'Energy efficiency reduces fossil fuel use in industry. '
            'Policy incentives affect renewable investment decisions. '
            'Fossil fuels remain common in some regions.'
        )
    },
    {
        'id': 205,
        'title': 'Taj Mahal and Paris',
        'body': (
            'The Taj Mahal was built by Emperor Shah Jahan in Agra. '
            'The monument attracts millions of tourists every year. '
            'Paris is the capital city of France. '
            'The Eiffel Tower attracts visitors from around the world.'
        )
    },
    {
        'id': 206,
        'title': 'Historic monuments',
        'body': (
            'Agra is a city known for historic monuments. '
            'Many emperors commissioned architecture across India. '
            'Tourism supports local economies.'
        )
    }
]

comparison_queries = [
    {
        'query': 'What computes gradients for neural networks?',
        'answer': 'backpropagation',
        'doc_id': 201
    },
    {
        'query': 'What reduces fossil fuel use?',
        'answer': 'solar and wind',
        'doc_id': 203
    },
    {
        'query': 'Who built the Taj Mahal?',
        'answer': 'Shah Jahan',
        'doc_id': 205
    }
]

fixed_chunker_eval = lambda text: fixed_size_chunking(text, chunk_size=10, overlap=0)
sentence_chunker_eval = lambda text: sentence_aware_chunking(text, max_words=20, overlap_sentences=1)
semantic_chunker_eval = lambda text: semantic_chunking(text, similarity_threshold=0.55, min_sentences=1, max_words=50)

baseline = evaluate_chunking('Fixed-size (baseline)', fixed_chunker_eval, comparison_docs, comparison_queries)
sentence = evaluate_chunking('Sentence-aware', sentence_chunker_eval, comparison_docs, comparison_queries)
semantic = evaluate_chunking('Semantic', semantic_chunker_eval, comparison_docs, comparison_queries)

results = [baseline, sentence, semantic]

print('Baseline vs improved chunking (fixed-size baseline)')
print('Method | Top1 | Top3 | MRR | Delta vs baseline')
for r in results:
    if r is baseline:
        delta = '---'
    else:
        delta = (
            f"{r['top1'] - baseline['top1']:+.2f}/"
            f"{r['top3'] - baseline['top3']:+.2f}/"
            f"{r['mrr'] - baseline['mrr']:+.2f}"
        )
    print(f"{r['method']:<18} | {r['top1']:.2f} | {r['top3']:.2f} | {r['mrr']:.2f} | {delta}")

print('\nTradeoff snapshot (no universal winner)')
print('Method | Chunks | Avg words')
for r in results:
    print(f"{r['method']:<18} | {r['chunks']:>6} | {r['avg_len']:>9.1f}")

Baseline vs improved chunking (fixed-size baseline)
Method | Top1 | Top3 | MRR | Delta vs baseline
Fixed-size (baseline) | 0.33 | 0.67 | 0.44 | ---
Sentence-aware     | 0.67 | 1.00 | 0.78 | +0.33/+0.33/+0.33
Semantic           | 0.67 | 0.67 | 0.67 | +0.33/+0.00/+0.22

Tradeoff snapshot (no universal winner)
Method | Chunks | Avg words
Fixed-size (baseline) |     19 |       8.9
Sentence-aware     |     14 |      16.6
Semantic           |     27 |       6.3


Relative to the fixed-size baseline, sentence-aware improves Top1/Top3/MRR by +0.33/+0.33/+0.33, while semantic improves Top1 and MRR but keeps Top3 flat. The tradeoff snapshot shows sentence-aware yields fewer, larger chunks, while semantic yields more, smaller chunks.

## Embedding models and retrieval patterns

Embedding model selection:
- **OpenAI:** strong English performance and easy integration.
- **Cohere:** good for multi-language and messy text.
- **Open-source:** best for data sovereignty, offline use, or cost control.

Retrieval patterns:
- **Top-k:** always returns k results, even if relevance is weak.
- **Similarity threshold:** filters low-quality matches at the cost of sometimes returning none.

Grounding reduces hallucinations by tying answers to retrieved evidence instead of model memory alone.

In [46]:
threshold_chunker = lambda text: sentence_aware_chunking(text, max_words=40, overlap_sentences=1)

threshold_chunks = build_chunks(ingested_docs, threshold_chunker)
threshold_vectorizer = SimpleTfidfVectorizer()
threshold_vectors = threshold_vectorizer.fit_transform([c['text'] for c in threshold_chunks])

def retrieve_with_threshold(query, chunks, vectorizer, chunk_vectors, threshold=0.20, max_k=5):
    query_vec = vectorizer.transform([query])[0]
    scores = cosine_similarity_scores(query_vec, chunk_vectors)
    filtered = [(i, s) for i, s in enumerate(scores) if s >= threshold]
    filtered.sort(key=lambda x: x[1], reverse=True)
    filtered = filtered[:max_k]
    return [chunks[i] for i, _ in filtered]

def eval_threshold(threshold):
    hits = 0
    retrieved_counts = []
    for item in corpus_queries:
        retrieved = retrieve_with_threshold(
            item['query'],
            threshold_chunks,
            threshold_vectorizer,
            threshold_vectors,
            threshold=threshold,
            max_k=5
        )
        retrieved_counts.append(len(retrieved))
        if any(item['answer'].lower() in c['text'].lower() for c in retrieved):
            hits += 1
    hit_rate = hits / len(corpus_queries)
    avg_returned = float(np.mean(retrieved_counts))
    return hit_rate, avg_returned

def eval_top_k(k):
    hits = 0
    retrieved_counts = []
    for item in corpus_queries:
        retrieved = retrieve_top_k(
            item['query'],
            threshold_chunks,
            threshold_vectorizer,
            threshold_vectors,
            top_k=k
        )
        retrieved_counts.append(len(retrieved))
        if any(item['answer'].lower() in c['text'].lower() for c in retrieved):
            hits += 1
    hit_rate = hits / len(corpus_queries)
    avg_returned = float(np.mean(retrieved_counts))
    return hit_rate, avg_returned

topk_hit, topk_avg = eval_top_k(3)
threshold_hit, threshold_avg = eval_threshold(0.20)

print('Retrieval pattern comparison')
print('Pattern | Hit rate | Avg returned')
print(f'Top-k (k=3) | {topk_hit:.2f} | {topk_avg:.2f}')
print(f'Threshold (0.20) | {threshold_hit:.2f} | {threshold_avg:.2f}')

Retrieval pattern comparison
Pattern | Hit rate | Avg returned
Top-k (k=3) | 1.00 | 3.00
Threshold (0.20) | 1.00 | 2.00


In [47]:
thresholds = [0.05, 0.10, 0.20, 0.30, 0.40]

print('Threshold sweep')
print('Threshold | Hit rate | Avg returned')
for t in thresholds:
    hit_rate, avg_returned = eval_threshold(t)
    print(f'{t:>8.2f} | {hit_rate:.2f} | {avg_returned:.2f}')

Threshold sweep
Threshold | Hit rate | Avg returned
    0.05 | 1.00 | 4.42
    0.10 | 1.00 | 3.50
    0.20 | 1.00 | 2.00
    0.30 | 1.00 | 1.00
    0.40 | 1.00 | 1.00


As the threshold rises, fewer chunks are returned while the hit rate stays at 1.00 here. This illustrates the precision-recall tradeoff: higher thresholds reduce noise but can risk missing answers on tougher corpora.
The threshold filter keeps the hit rate at 1.00 while returning fewer chunks on average (2 vs 3), showing how similarity thresholds can cut noise without sacrificing coverage at a well-chosen cutoff.

## Query rewriting

There is no single best technique. Each method supports a different precision vs recall goal, and real systems often combine them. In the lab below, each method is compared against its own baseline (no rewrite) to show improvement rather than to rank methods.

1. Query segmentation
- What it does: breaks a query into semantic phrases.
- Best used for: improving precision by treating key phrases as a unit.

2. Query scoping
- What it does: restricts matching to specific fields (title, tags, or metadata).
- Best used for: aggressive precision to avoid irrelevant matches.

3. Query relaxation
- What it does: removes optional terms when results are too narrow.
- Best used for: increasing recall so users still get results.

4. Query expansion
- What it does: adds synonyms or related terms to reduce vocabulary mismatch.
- Best used for: improving recall when the corpus uses different wording.

Recommended pipeline:
- Segment first so the system understands core phrases.
- Scope second to enforce where those phrases must match.
- Expand or relax last if results are too narrow.

In [ ]:
# Ordered to surface baseline errors for segmentation/scoping.
rewrite_docs = {
    1: {'title': 'Intellectual Property Attorney', 'body': 'An attorney specializing in intellectual property law.'},
    2: {'title': 'IP Networking Basics', 'body': 'Internet protocol and computer networking concepts.'},
    3: {'title': 'Cute Fluffy Kittens', 'body': 'Photos of fluffy kittens and cute cats.'},
    4: {'title': 'Fluffy Kittens Playing', 'body': 'Small fluffy kittens playing together.'},
    6: {'title': 'White Shirt Dress', 'body': 'Elegant white shirt dress for women.'},
    5: {'title': 'White Dress Shirt', 'body': 'Formal white dress shirt for office wear.'},
    9: {'title': 'Artificial Intelligence', 'body': 'Machine learning is widely used in AI.'},
    7: {'title': 'Machine Learning', 'body': 'Introduction to machine learning algorithms.'},
    8: {'title': 'Learning Techniques', 'body': 'Different educational learning methods.'},
    10: {'title': 'Dress Collection', 'body': 'Women dress collection and fashion.'},
    11: {'title': 'Cute Puppies', 'body': 'Cute dogs and puppies.'}
}

expansion_queries = [
    {'query': 'ip lawer', 'doc_id': 1},
    {'query': 'ml algorithms', 'doc_id': 7}
]
relaxation_queries = [
    {'query': 'cute fluffy kittens', 'doc_id': 3},
    {'query': 'tiny fluffy kittens playing', 'doc_id': 4}
]
segmentation_queries = [
    {'query': 'white dress shirt', 'doc_id': 5}
]
scoping_queries = [
    {'query': 'machine learning', 'doc_id': 7},
    {'query': 'learning techniques', 'doc_id': 8}
]

SYNONYMS = {
    'ip': ['intellectual property'],
    'lawer': ['lawyer', 'attorney'],
    'ml': ['machine learning']
}
RELAX_TERMS = {'tiny'}
PHRASES = ['dress shirt', 'intellectual property', 'machine learning']

def lexical_search(terms, docs, phrase=None, strict_and=False, scope='all'):
    scores = {}
    for doc_id, content in docs.items():
        title = content['title'].lower()
        body = content['body'].lower()
        if scope == 'title':
            searchable = title
        elif scope == 'body':
            searchable = body
        else:
            searchable = title + ' ' + body
        matched = 0
        for term in terms:
            if term in searchable:
                matched += 1
                scores[doc_id] = scores.get(doc_id, 0) + 1
        if strict_and and matched < len(terms):
            scores.pop(doc_id, None)
            continue
        if phrase and phrase in searchable:
            scores[doc_id] = scores.get(doc_id, 0) + 5
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in ranked]

def run_retrieval(query_text, opts=None):
    if opts is None:
        terms = tokenize(query_text)
        phrase = None
        strict_and = False
        scope = 'all'
    else:
        terms = opts.get('terms', tokenize(query_text))
        phrase = opts.get('phrase')
        strict_and = opts.get('strict_and', False)
        scope = opts.get('scope', 'all')
    return lexical_search(terms, rewrite_docs, phrase=phrase, strict_and=strict_and, scope=scope)

def rewrite_expansion(query_text):
    terms = tokenize(query_text)
    expanded = []
    for term in terms:
        expanded.append(term)
        for syn in SYNONYMS.get(term, []):
            expanded.extend(tokenize(syn))
    return {'terms': expanded, 'strict_and': False}

def rewrite_relaxation(query_text):
    terms = [t for t in tokenize(query_text) if t not in RELAX_TERMS]
    return {'terms': terms, 'strict_and': False}

def rewrite_segmentation(query_text):
    phrase = None
    lowered = query_text.lower()
    for p in PHRASES:
        if p in lowered:
            phrase = p
            break
    return {'terms': tokenize(query_text), 'phrase': phrase, 'strict_and': False}

def rewrite_scoping(query_text):
    return {'terms': tokenize(query_text), 'scope': 'title', 'strict_and': False}

def evaluate_lexical(queries, opts_fn=None):
    top1 = 0
    top3 = 0
    for item in queries:
        opts = opts_fn(item['query']) if opts_fn else None
        results = run_retrieval(item['query'], opts)
        if results and results[0] == item['doc_id']:
            top1 += 1
        if item['doc_id'] in results[:3]:
            top3 += 1
    total = len(queries)
    return {
        'top1': top1 / total if total else 0.0,
        'top3': top3 / total if total else 0.0
    }

In [ ]:
groups = [
    ('Expansion', expansion_queries, lambda q: {'terms': tokenize(q), 'strict_and': True}, rewrite_expansion),
    ('Relaxation', relaxation_queries, lambda q: {'terms': tokenize(q), 'strict_and': True}, rewrite_relaxation),
    ('Segmentation', segmentation_queries, lambda q: {'terms': tokenize(q), 'strict_and': False}, rewrite_segmentation),
    ('Scoping', scoping_queries, lambda q: {'terms': tokenize(q), 'strict_and': False}, rewrite_scoping)
]

print('Query rewriting evaluation')
print('Method | Top1 before | Top1 after | Top3 before | Top3 after')
for label, queries, baseline_fn, rewrite_fn in groups:
    before = evaluate_lexical(queries, baseline_fn)
    after = evaluate_lexical(queries, rewrite_fn)
    print(f"{label:<11} | {before['top1']:.2f} | {after['top1']:.2f} | {before['top3']:.2f} | {after['top3']:.2f}")

Query rewriting evaluation
Method | Top1 before | Top1 after | Top3 before | Top3 after
Expansion   | 0.00 | 1.00 | 0.00 | 1.00
Relaxation  | 0.50 | 1.00 | 0.50 | 1.00
Segmentation | 0.00 | 1.00 | 1.00 | 1.00
Scoping     | 0.50 | 1.00 | 1.00 | 1.00


Each method improves over its own baseline, not over the other methods. Expansion and relaxation primarily boost recall, while segmentation and scoping tighten precision when queries are ambiguous.

## Lab 2: RAG pipeline build

Build a full pipeline: ingest -> chunk -> embed -> retrieve -> grounded generate. Then compare chunk size variants on retrieval quality.

In [ ]:
def build_rag_index(docs, chunk_fn):
    chunks = build_chunks(docs, chunk_fn)
    vectorizer = SimpleTfidfVectorizer()
    vectors = vectorizer.fit_transform([c['text'] for c in chunks])
    return chunks, vectorizer, vectors

def generate_answer(query, retrieved_chunks):
    query_terms = set(tokenize(query))
    candidates = []
    for chunk in retrieved_chunks:
        for sentence in split_sentences(chunk['text']):
            score = sum(1 for t in tokenize(sentence) if t in query_terms)
            if score > 0:
                candidates.append((score, sentence))
    if not candidates:
        return retrieved_chunks[0]['text'] if retrieved_chunks else ''
    candidates.sort(key=lambda x: x[0], reverse=True)
    top_sentences = [s for _, s in candidates[:2]]
    return ' '.join(top_sentences)

In [ ]:
small_chunker = lambda text: fixed_size_chunking(text, chunk_size=8, overlap=0)
large_chunker = lambda text: fixed_size_chunking(text, chunk_size=40, overlap=5)

small = evaluate_chunking('Fixed-8', small_chunker, ingested_docs, corpus_queries)
large = evaluate_chunking('Fixed-40', large_chunker, ingested_docs, corpus_queries)

print('Chunk size comparison (same method, different sizes)')
print('Method | Chunks | Avg words | Top1 | Top3 | MRR')
for r in [small, large]:
    print(f"{r['method']:<8} | {r['chunks']:>6} | {r['avg_len']:>9.1f} | {r['top1']:.2f} | {r['top3']:.2f} | {r['mrr']:.2f}")

Chunk size comparison (same method, different sizes)
Method | Chunks | Avg words | Top1 | Top3 | MRR
Fixed-8  |    118 |       6.8 | 0.58 | 0.83 | 0.71
Fixed-40 |     60 |      13.4 | 1.00 | 1.00 | 1.00


Using the smaller fixed-size chunks as the baseline, the larger chunk size improves Top1/Top3/MRR and restores context. This reinforces that chunk size tuning is a baseline-driven improvement, not a universal winner.

In [ ]:
sample_query = corpus_queries[0]['query']
chunks, vectorizer, vectors = build_rag_index(ingested_docs, sentence_chunker)
retrieved = retrieve_top_k(sample_query, chunks, vectorizer, vectors, top_k=3)
answer = generate_answer(sample_query, retrieved)

print(f'Query: {sample_query}')
for i, chunk in enumerate(retrieved, 1):
    snippet = chunk['text'][:160].replace('\n', ' ')
    print(f"{i}. Doc {chunk['doc_id']} - {chunk['title']}: {snippet}...")
print(f'Generated answer: {answer}')

Query: What computes gradients for neural networks?
1. Doc 1 - Backpropagation and gradients: Backpropagation and gradients. Backpropagation computes gradients for neural networks during training....
2. Doc 16 - Ai Notes 04: Ai Notes 04. Feature scaling can improve optimization. Neural networks learn patterns from data....
3. Doc 20 - Ai Notes 08: Ai Notes 08. Feature scaling can improve optimization. Neural networks learn patterns from data....
Generated answer: Backpropagation computes gradients for neural networks during training. Neural networks learn patterns from data.


The top chunk contains the exact evidence sentence, and the generated answer cites it directly, showing grounded generation from retrieved context.